# Whisper Small LoRA Fine-Tuning for Nepali Speech Recognition

This notebook records a LoRA fine-tuning experiment using the pretrained Whisper Small model.

LoRA trains a small set of additional adapter parameters instead of updating all the model parameters. The goal here is to adapt Whisper Small for Nepali speech recognition while using less GPU memory than full fine-tuning.

The notebook covers dataset preparation, LoRA setup, training, validation, test inference, and error analysis. The original code outputs have been retained for the cells kept in this version. The notebook has not been rerun.

## 1. Google Drive and environment setup

Mount Google Drive to access the dataset, saved checkpoints, and result files. The following cells also check the installed PyTorch and PEFT versions and confirm whether CUDA and a GPU are available.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
import peft
from importlib.util import find_spec

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("torchao installed:", find_spec("torchao") is not None)

PyTorch: 2.14.1+cu130
PEFT: 0.21.2
torchao installed: False


In [ ]:
import torch
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Main project directory
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Nepali_Whisper_Project"
)

# Dataset manifests
TRAIN_CSV = PROJECT_DIR / "train_manifest.csv"
VAL_CSV = PROJECT_DIR / "validation_manifest.csv"
TEST_CSV = PROJECT_DIR / "test_manifest.csv"

# Audio directories
TRAIN_AUDIO_DIR = PROJECT_DIR / "audio" / "train"
VAL_AUDIO_DIR = PROJECT_DIR / "audio" / "validation"
TEST_AUDIO_DIR = PROJECT_DIR / "audio" / "test"

In [ ]:
# LoRA experiment directory
LORA_DIR = PROJECT_DIR / "whisper_small_lora"

CHECKPOINT_DIR = LORA_DIR / "checkpoints"
FINAL_MODEL_DIR = LORA_DIR / "final_model"
RESULTS_DIR = LORA_DIR / "results"

# Create directories
for directory in [
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
    RESULTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("LoRA directories created successfully!")

LoRA directories created successfully!


In [ ]:
print("===== ENVIRONMENT =====")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print("GPU memory:", round(gpu_memory, 2), "GB")


print("\n===== LIBRARY VERSIONS =====")

for package in [
    "transformers",
    "peft",
    "accelerate",
    "datasets"
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


print("\n===== DATASET PATHS =====")

for path in [
    TRAIN_CSV,
    VAL_CSV,
    TEST_CSV,
    TRAIN_AUDIO_DIR,
    VAL_AUDIO_DIR,
    TEST_AUDIO_DIR
]:
    status = "OK" if path.exists() else "MISSING"
    print(f"{path.name}: {status}")

===== ENVIRONMENT =====
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

===== LIBRARY VERSIONS =====
transformers: 5.22.1
peft: 0.21.2
accelerate: 1.15.0
datasets: 4.8.5

===== DATASET PATHS =====
train_manifest.csv: OK
validation_manifest.csv: OK
test_manifest.csv: OK
train: OK
validation: OK
test: OK


## 2. Load and inspect the dataset

The train, validation, and test manifests contain audio file IDs and their corresponding transcripts. These checks look at the dataset sizes, missing values, and duplicate IDs before training.

In [ ]:
import pandas as pd

# Load CSV files
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

# Display dataset sizes
print("===== DATASET SIZES =====")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

# Inspect training data
print("\n===== TRAINING DATA =====")
display(train_df.head())

# Inspect columns
print("\nColumns:", train_df.columns.tolist())

===== DATASET SIZES =====
Training: 34941
Validation: 7622
Testing: 7495

===== TRAINING DATA =====


,file_id,archive_prefix,archive_path,file_size,compressed_size,speaker_id,transcription,split
0,003017eaca,0,asr_nepali/data/00/003017eaca.flac,49947,49947,97b71,थिए तर पछि,train
1,00ad922f03,0,asr_nepali/data/00/00ad922f03.flac,69904,69904,55c0f,४ ओभरमा १ सय ३२,train
2,00c3495e71,0,asr_nepali/data/00/00c3495e71.flac,63768,63768,e2039,यसको भूमिका धेरै,train
3,00615be9e8,0,asr_nepali/data/00/00615be9e8.flac,81609,81609,2001f,लगातार जारी औद्योगिक,train
4,009df49393,0,asr_nepali/data/00/009df49393.flac,72985,72985,fad24,चन्द्रको नेतृत्वमा सिपाहीहरू,train



Columns: ['file_id', 'archive_prefix', 'archive_path', 'file_size', 'compressed_size', 'speaker_id', 'transcription', 'split']


In [ ]:
# Check missing values
print("===== MISSING VALUES =====")
print(train_df.isnull().sum())

# Check duplicate file IDs
print("\n===== DUPLICATE FILE IDs =====")
print("Train:", train_df["file_id"].duplicated().sum())
print("Validation:", val_df["file_id"].duplicated().sum())
print("Test:", test_df["file_id"].duplicated().sum())

# Check speaker overlap
train_speakers = set(train_df["speaker_id"])
val_speakers = set(val_df["speaker_id"])
test_speakers = set(test_df["speaker_id"])

print("\n===== SPEAKER OVERLAP =====")
print("Train-Val:", len(train_speakers & val_speakers))
print("Train-Test:", len(train_speakers & test_speakers))
print("Val-Test:", len(val_speakers & test_speakers))

# Check empty transcriptions
print("\n===== EMPTY TRANSCRIPTIONS =====")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    empty = df["transcription"].fillna("").str.strip().eq("").sum()
    print(f"{name}: {empty}")

===== MISSING VALUES =====
file_id            0
archive_prefix     0
archive_path       0
file_size          0
compressed_size    0
speaker_id         0
transcription      0
split              0
dtype: int64

===== DUPLICATE FILE IDs =====
Train: 0
Validation: 0
Test: 0

===== SPEAKER OVERLAP =====
Train-Val: 0
Train-Test: 0
Val-Test: 0

===== EMPTY TRANSCRIPTIONS =====
Train: 0
Validation: 0
Test: 0


## 3. Load the Whisper processor and prepare the audio

The processor prepares audio features and tokenizes the transcripts for Whisper. Audio is loaded as mono at 16 kHz, which is the sample rate expected by the model.

The sample check below is a quick way to confirm that the audio and transcript preprocessing functions are working before creating the full datasets.

In [ ]:
from transformers import WhisperProcessor

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="nepali",
    task="transcribe"
)

print("Whisper processor loaded successfully!")
print("Tokenizer vocabulary size:", processor.tokenizer.vocab_size)
print("Feature extractor sampling rate:",
      processor.feature_extractor.sampling_rate)

Whisper processor loaded successfully!
Tokenizer vocabulary size: 50258
Feature extractor sampling rate: 16000


In [ ]:
import librosa
import numpy as np

def prepare_audio(file_id, audio_dir):
  audio_path = audio_dir / f"{file_id}.flac"

  #Load audio at 16 kHz
  audio, _ = librosa.load(
      audio_path,
      sr=16000,
      mono=True
  )

  # Extract log-mel spectrogram
  input_features = processor.feature_extractor(
      audio,
      sampling_rate=16000
  ).input_features[0]

  return input_features

def prepare_text(transcription):
  labels = processor.tokenizer(
      transcription
  ).input_ids

  return labels

In [ ]:
# Select one training sample
sample = train_df.iloc[0]

# Process audio
sample_features = prepare_audio(
    sample["file_id"],
    TRAIN_AUDIO_DIR
)

# Process transcription
sample_labels = prepare_text(
    sample["transcription"]
)

print("===== AUDIO FEATURES =====")
print("Shape:", sample_features.shape)
print("Data type:", sample_features.dtype)

print("\n===== TEXT LABELS =====")
print("Token IDs:", sample_labels)
print("Number of tokens:", len(sample_labels))

print("\nOriginal transcription:", sample["transcription"])
print("Decoded transcription:",
      processor.tokenizer.decode(sample_labels))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


===== AUDIO FEATURES =====
Shape: (80, 3000)
Data type: float32

===== TEXT LABELS =====
Token IDs: [50258, 50313, 50359, 50363, 3941, 98, 33279, 3941, 237, 8485, 97, 25411, 8485, 103, 3941, 249, 33279, 50257]
Number of tokens: 18

Original transcription: थिए तर पछि
Decoded transcription: <|startoftranscript|><|ne|><|transcribe|><|notimestamps|>थिए तर पछि<|endoftext|>


## 4. Create the datasets and data collator

The dataset class loads and processes an audio sample when it is requested. Separate dataset objects are created for training, validation, and testing.

The data collator combines individual samples into a batch. It pads transcript sequences and uses `-100` for padding positions so that those positions are ignored when the loss is calculated.

In [ ]:
import torch
from torch.utils.data import Dataset

class WhisperDataset(Dataset):

    def __init__(self, dataframe, audio_dir):
        self.dataframe = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Process audio
        input_features = prepare_audio(
            row["file_id"],
            self.audio_dir
        )

        # Process transcription
        labels = prepare_text(
            row["transcription"]
        )

        return {
            "input_features": torch.tensor(
                input_features,
                dtype=torch.float32
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }

In [ ]:
# Create a small dataset using the first 5 samples
sample_dataset = WhisperDataset(
    train_df.head(5),
    TRAIN_AUDIO_DIR
)

# Retrieve one sample
sample_item = sample_dataset[0]

print("===== SAMPLE DATASET =====")
print("Dataset size:", len(sample_dataset))

print("\n===== AUDIO FEATURES =====")
print("Shape:", sample_item["input_features"].shape)
print("Data type:", sample_item["input_features"].dtype)

print("\n===== LABELS =====")
print("Shape:", sample_item["labels"].shape)
print("Data type:", sample_item["labels"].dtype)

print("\nDecoded text:")
print(
    processor.tokenizer.decode(
        sample_item["labels"],
        clean_up_tokenization_spaces=False
    )
)

===== SAMPLE DATASET =====
Dataset size: 5

===== AUDIO FEATURES =====
Shape: torch.Size([80, 3000])
Data type: torch.float32

===== LABELS =====
Shape: torch.Size([18])
Data type: torch.int64

Decoded text:
<|startoftranscript|><|ne|><|transcribe|><|notimestamps|>थिए तर पछि<|endoftext|>


In [ ]:
train_dataset = WhisperDataset(
    train_df,
    TRAIN_AUDIO_DIR
)

val_dataset = WhisperDataset(
    val_df,
    VAL_AUDIO_DIR
)

test_dataset = WhisperDataset(
    test_df,
    TEST_AUDIO_DIR
)

print("===== DATASET SIZES =====")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

===== DATASET SIZES =====
Training: 34941
Validation: 7622
Testing: 7495


In [ ]:
from dataclasses import dataclass

@dataclass
class WhisperDataCollator:
    decoder_start_token_id: int

    def __call__(self, features):
        # Remove decoder-start token from each label sequence
        label_features = []

        for feature in features:
            labels = feature["labels"]

            if labels[0].item() == self.decoder_start_token_id:
                labels = labels[1:]

            label_features.append(labels)

        # Stack audio features
        input_features = torch.stack([
            feature["input_features"] for feature in features
        ])

        # Pad labels to the longest sequence
        labels = torch.nn.utils.rnn.pad_sequence(
            label_features,
            batch_first=True,
            padding_value=-100
        )

        return {
            "input_features": input_features,
            "labels": labels
        }


data_collator = WhisperDataCollator(
    decoder_start_token_id=model.config.decoder_start_token_id
)


In [ ]:
# Retrieve two samples
sample_1 = train_dataset[0]
sample_2 = train_dataset[1]

# Combine them into a batch
batch = data_collator([sample_1, sample_2])

print("===== BATCH SHAPES =====")

print(
    "Input features:",
    batch["input_features"].shape
)

print(
    "Labels:",
    batch["labels"].shape
)

print("\n===== LABEL PADDING =====")
print("Sample 1 label length:", len(sample_1["labels"]))
print("Sample 2 label length:", len(sample_2["labels"]))

print(
    "Number of padding tokens:",
    (batch["labels"] == -100).sum().item()
)

===== BATCH SHAPES =====
Input features: torch.Size([2, 80, 3000])
Labels: torch.Size([2, 23])

===== LABEL PADDING =====
Sample 1 label length: 18
Sample 2 label length: 24
Number of padding tokens: 6


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=True
)

print("===== DATALOADER INFORMATION =====")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))
print("Batch size:", BATCH_SIZE)

===== DATALOADER INFORMATION =====
Training batches: 17471
Validation batches: 3811
Testing batches: 3748
Batch size: 2


## 5. Load Whisper Small and add LoRA

First, the pretrained Whisper Small model is loaded. LoRA is then added to selected model layers using PEFT.

Only the LoRA adapter parameters are intended to be trainable. This reduces the number of parameters updated during training compared with full fine-tuning.

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    attn_implementation="sdpa"
)

print("===== MODEL INFORMATION =====")
print("Model:", MODEL_NAME)
print("Decoder start token ID:", model.config.decoder_start_token_id)
print("Vocabulary size:", model.config.vocab_size)

print("\n===== LABEL TOKEN CHECK =====")
print("First label token:", train_dataset[0]["labels"][0].item())
print(
    "Decoder start token:",
    model.config.decoder_start_token_id
)

print("\nModel loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

===== MODEL INFORMATION =====
Model: openai/whisper-small
Decoder start token ID: 50258
Vocabulary size: 51865

===== LABEL TOKEN CHECK =====
First label token: 50258
Decoder start token: 50258

Model loaded successfully!


In [ ]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

# Reload the original pretrained Whisper Small
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    attn_implementation="sdpa"
)

# Configure LoRA without the Seq2Seq task wrapper
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

# Apply generic PEFT wrapper
model = get_peft_model(base_model, lora_config)

# Configure memory optimization
model.base_model.model.config.use_cache = False
model.base_model.model.gradient_checkpointing_enable()
model.base_model.model.enable_input_require_grads()

print("Model recreated successfully!")
model.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Model recreated successfully!
trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267


In [ ]:
# Disable KV caching during training
model.config.use_cache = False

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Ensure input embeddings can receive gradients
model.enable_input_require_grads()

print("===== TRAINING CONFIGURATION =====")
print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("KV cache:", model.config.use_cache)
print("Trainable parameters: 1,769,472")

===== TRAINING CONFIGURATION =====
Gradient checkpointing: True
KV cache: False
Trainable parameters: 1,769,472


## 6. Configure training

The training arguments define the number of epochs, batch size, gradient accumulation, learning rate, precision, and checkpoint saving frequency.

Gradient accumulation helps increase the effective batch size without needing to fit the full batch into GPU memory at once. FP16 and gradient checkpointing are used to reduce memory use.

The trainer connects the model, training arguments, training dataset, and data collator.

In [ ]:
import inspect
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

print("===== TRAINING ARGUMENTS =====")

training_params = inspect.signature(
    Seq2SeqTrainingArguments.__init__
).parameters

for param in [
    "output_dir",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "num_train_epochs",
    "warmup_steps",
    "fp16",
    "gradient_checkpointing",
    "eval_strategy",
    "save_strategy",
    "logging_steps",
    "save_steps",
    "report_to",
    "remove_unused_columns",
    "predict_with_generate"
]:
    print(f"{param}: {param in training_params}")

print("\n===== TRAINER ARGUMENTS =====")

trainer_params = inspect.signature(
    Seq2SeqTrainer.__init__
).parameters

for param in [
    "model",
    "args",
    "train_dataset",
    "eval_dataset",
    "data_collator",
    "processing_class",
    "tokenizer"
]:
    print(f"{param}: {param in trainer_params}")

===== TRAINING ARGUMENTS =====
output_dir: True
per_device_train_batch_size: True
gradient_accumulation_steps: True
learning_rate: True
num_train_epochs: True
warmup_steps: True
fp16: True
gradient_checkpointing: True
eval_strategy: True
save_strategy: True
logging_steps: True
save_steps: True
report_to: True
remove_unused_columns: True
predict_with_generate: True

===== TRAINER ARGUMENTS =====
model: True
args: True
train_dataset: True
eval_dataset: True
data_collator: True
processing_class: True
tokenizer: False


In [ ]:
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR = str(CHECKPOINT_DIR)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training duration
    num_train_epochs=1,

    # Batch configuration
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimization
    learning_rate=1e-4,
    warmup_steps=100,
    weight_decay=0.01,

    # Precision and memory
    fp16=True,
    gradient_checkpointing=True,

    # Logging and checkpoints
    logging_steps=100,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    # Evaluation will be performed separately
    eval_strategy="no",

    # Other settings
    predict_with_generate=False,
    remove_unused_columns=False,
    report_to="none",

    # Reproducibility
    seed=42,
    data_seed=42
)

print("LoRA training arguments configured!")
print("Effective batch size:",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)
print("Total optimizer steps:",
      len(train_loader) // training_args.gradient_accumulation_steps)

LoRA training arguments configured!
Effective batch size: 16
Total optimizer steps: 2183


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor
)

print("Trainer recreated successfully!")
print("Training samples:", len(train_dataset))
print("Optimizer steps:", 2183)
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))

Trainer recreated successfully!
Training samples: 34941
Optimizer steps: 2183
Trainable parameters: 1769472


## 7. Check the training step and resume from checkpoint

Before continuing training, a forward and backward pass is tested on one batch. This checks that the model can calculate a loss and that the trainable LoRA parameters receive gradients.

The training cell then resumes from the saved `checkpoint-2000` checkpoint. The final adapter and processor are saved after training completes.

In [ ]:
# Get one training batch
batch = next(iter(train_loader))

# Move batch to GPU
batch = {
    key: value.to("cuda")
    for key, value in batch.items()
}

model.train()

# Forward pass
outputs = model(**batch)

loss = outputs.loss

print("===== FORWARD PASS =====")
print("Loss:", loss.item())

# Backward pass
loss.backward()

# Check LoRA gradients
gradient_count = sum(
    1 for name, parameter in model.named_parameters()
    if parameter.requires_grad and parameter.grad is not None
)

print("\n===== BACKWARD PASS =====")
print("LoRA parameters with gradients:", gradient_count)
print("Total trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

# Clear gradients after testing
model.zero_grad(set_to_none=True)

print("\nForward and backward passes successful!")

===== FORWARD PASS =====
Loss: 1.3619236946105957

===== BACKWARD PASS =====
LoRA parameters with gradients: 144
Total trainable parameters: 1769472

Forward and backward passes successful!


In [ ]:
import time
import torch

# Path to the saved checkpoint
CHECKPOINT_PATH = str(
    CHECKPOINT_DIR / "checkpoint-2000"
)

# Verify checkpoint exists
from pathlib import Path

assert Path(CHECKPOINT_PATH).exists(), "Checkpoint not found!"

print("Resuming from:", CHECKPOINT_PATH)

model.zero_grad(set_to_none=True)

print("===== RESUMING LoRA TRAINING =====")

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=CHECKPOINT_PATH
)

training_time = time.time() - start_time

print("\n===== LoRA TRAINING COMPLETED =====")
print(f"Training time: {training_time / 60:.2f} minutes")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training steps: {train_result.global_step}")

# Save final adapter
trainer.save_model(str(FINAL_MODEL_DIR))
processor.save_pretrained(str(FINAL_MODEL_DIR))

print("Final LoRA adapter saved:", FINAL_MODEL_DIR)

Resuming from: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_lora/checkpoints/checkpoint-2000
===== RESUMING LoRA TRAINING =====


[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	save_steps: 250 (from args) != 500 (from trainer_state.json)


Step,Training Loss
2100,2.299823



===== LoRA TRAINING COMPLETED =====
Training time: 26.88 minutes
Training loss: 2.2750
Training steps: 2184
Final LoRA adapter saved: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_lora/final_model


### Resuming LoRA Training

Training was resumed from the saved `checkpoint-2000` checkpoint instead of starting from the beginning. This allowed the experiment to continue from the previously completed training steps.

**Note about the warning:**

During resumption, Transformers displayed a warning because the current training configuration uses `save_steps=250`, while the checkpoint's saved training state contains `save_steps=500`.

This is a difference in checkpoint-saving frequency. It does not indicate that the model weights or training process are corrupted. The training continued successfully despite this warning.

**Training time:**

The resumed training took approximately **26.88 minutes** to complete.

Although the progress bar shows 2,184/2,184 steps, training was resumed from checkpoint-2000. Therefore, the remaining training steps were completed rather than repeating the entire training process.

The final reported training loss was **2.2750**, and the completed LoRA adapter was saved to the `final_model` directory.

## 8. Load the trained adapter and run validation inference

The trained LoRA adapter is loaded on top of the original Whisper Small model. Validation audio is processed in batches and the model generates a transcript for each sample.

Predictions are saved to a CSV file periodically. This makes it possible to continue from saved predictions if the runtime stops before inference is complete.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

BASE_MODEL = "openai/whisper-small"

ADAPTER_DIR = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/final_model"
)

# Load processor
processor = WhisperProcessor.from_pretrained(BASE_MODEL)

# Load original pretrained Whisper Small
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)

# Attach trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model = model.to("cuda")
model.eval()

print("LoRA model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

LoRA model loaded successfully!
GPU: Tesla T4
Trainable parameters: 0
Total parameters: 243504384


In [ ]:
import pandas as pd
import os

PROJECT_DIR = "/content/drive/MyDrive/Nepali_Whisper_Project"

VALIDATION_CSV = os.path.join(
    PROJECT_DIR, "validation_manifest.csv"
)

validation_df = pd.read_csv(VALIDATION_CSV)

print("Validation samples:", len(validation_df))
print("\nColumns:")
print(validation_df.columns.tolist())

print("\nFirst 5 rows:")
display(validation_df.head())

Validation samples: 7622

Columns:
['file_id', 'archive_prefix', 'archive_path', 'file_size', 'compressed_size', 'speaker_id', 'transcription', 'split']

First 5 rows:


,file_id,archive_prefix,archive_path,file_size,compressed_size,speaker_id,transcription,split
0,00f146a59a,0,asr_nepali/data/00/00f146a59a.flac,49692,49692,bdfe3,नियमित बस उत्तर,validation
1,00a49d7c51,0,asr_nepali/data/00/00a49d7c51.flac,50325,50325,a3cce,मेरीको विरुद्धमा भएको,validation
2,0061189667,0,asr_nepali/data/00/0061189667.flac,95666,95666,6a6d1,करिब ३५ ४०,validation
3,001907805e,0,asr_nepali/data/00/001907805e.flac,82648,82648,f4b4a,दृष्टिकोणले किराँती विशेषता,validation
4,00a2b01b40,0,asr_nepali/data/00/00a2b01b40.flac,66386,66386,12f3b,चरा कहाँ कस्तो,validation


In [ ]:
import os

VALIDATION_AUDIO_DIR = os.path.join(
    PROJECT_DIR, "audio", "validation"
)

validation_df["audio_path"] = validation_df["file_id"].apply(
    lambda file_id: os.path.join(
        VALIDATION_AUDIO_DIR, f"{file_id}.flac"
    )
)

# Check whether audio files exist
validation_df["audio_exists"] = validation_df["audio_path"].apply(
    os.path.exists
)

print("Total validation samples:", len(validation_df))
print("Existing audio files:", validation_df["audio_exists"].sum())
print("Missing audio files:", (~validation_df["audio_exists"]).sum())

print("\nSample audio paths:")
display(validation_df[["file_id", "audio_path", "audio_exists"]].head())

Total validation samples: 7622
Existing audio files: 7622
Missing audio files: 0

Sample audio paths:


,file_id,audio_path,audio_exists
0,00f146a59a,/content/drive/MyDrive/Nepali_Whisper_Project/...,True
1,00a49d7c51,/content/drive/MyDrive/Nepali_Whisper_Project/...,True
2,0061189667,/content/drive/MyDrive/Nepali_Whisper_Project/...,True
3,001907805e,/content/drive/MyDrive/Nepali_Whisper_Project/...,True
4,00a2b01b40,/content/drive/MyDrive/Nepali_Whisper_Project/...,True


In [ ]:
import librosa
import numpy as np
import pandas as pd
import torch
import os
import time
from tqdm.auto import tqdm

RESULTS_DIR = os.path.join(
    PROJECT_DIR, "whisper_small_lora", "results"
)
os.makedirs(RESULTS_DIR, exist_ok=True)

PREDICTION_FILE = os.path.join(
    RESULTS_DIR, "validation_predictions.csv"
)

BATCH_SIZE = 8
CHECKPOINT_INTERVAL = 200

# Resume from an existing prediction checkpoint if available
if os.path.exists(PREDICTION_FILE):
    predictions_df = pd.read_csv(PREDICTION_FILE)
    completed_ids = set(predictions_df["file_id"].astype(str))
    print("Resuming from saved predictions:", len(completed_ids))
else:
    predictions_df = pd.DataFrame(
        columns=["file_id", "reference", "prediction"]
    )
    completed_ids = set()

remaining_df = validation_df[
    ~validation_df["file_id"].astype(str).isin(completed_ids)
].reset_index(drop=True)

print("Remaining samples:", len(remaining_df))

def predict_batch(batch):
    audio_arrays = []

    for path in batch["audio_path"]:
        audio, sr = librosa.load(path, sr=16000, mono=True)
        audio_arrays.append(audio)

    inputs = processor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device="cuda",
        dtype=torch.float16
    )

    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features,
            language="ne",
            task="transcribe",
            use_cache=True
        )

    return processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )

start_time = time.time()

new_predictions = []

for start in tqdm(range(0, len(remaining_df), BATCH_SIZE)):
    batch = remaining_df.iloc[start:start + BATCH_SIZE]

    batch_predictions = predict_batch(batch)

    for (_, row), prediction in zip(
        batch.iterrows(), batch_predictions
    ):
        new_predictions.append({
            "file_id": str(row["file_id"]),
            "reference": row["transcription"],
            "prediction": prediction
        })

    if len(new_predictions) >= CHECKPOINT_INTERVAL:
        predictions_df = pd.concat(
            [predictions_df, pd.DataFrame(new_predictions)],
            ignore_index=True
        )
        predictions_df.to_csv(PREDICTION_FILE, index=False)
        new_predictions = []

# Save remaining predictions
if new_predictions:
    predictions_df = pd.concat(
        [predictions_df, pd.DataFrame(new_predictions)],
        ignore_index=True
    )

predictions_df.to_csv(PREDICTION_FILE, index=False)

total_time = time.time() - start_time

print("\n===== LoRA VALIDATION INFERENCE COMPLETED =====")
print("Total predictions:", len(predictions_df))
print("Inference time:", round(total_time / 60, 2), "minutes")
print("Saved to:", PREDICTION_FILE)

Remaining samples: 7622


  0%|          | 0/953 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



===== LoRA VALIDATION INFERENCE COMPLETED =====
Total predictions: 7622
Inference time: 63.96 minutes
Saved to: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_lora/results/validation_predictions.csv


## 9. Evaluate validation predictions

References and predictions are normalized in the same way before calculating WER and CER. Punctuation is removed and whitespace is normalized.

WER measures word-level differences, while CER measures character-level differences. Exact matches are also counted. A small sample of predictions is displayed for manual inspection.

In [ ]:
!pip install -q jiwer

In [ ]:
import unicodedata
import re
import pandas as pd
from jiwer import wer, cer

# Text normalization
def normalize_text(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)

    # Remove punctuation
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Normalize references and predictions
predictions_df["ref_clean"] = (
    predictions_df["reference"].apply(normalize_text)
)

predictions_df["pred_clean"] = (
    predictions_df["prediction"].apply(normalize_text)
)

references = predictions_df["ref_clean"].tolist()
hypotheses = predictions_df["pred_clean"].tolist()

# Corpus-level metrics
validation_wer = wer(references, hypotheses)
validation_cer = cer(references, hypotheses)

# Exact matches
exact_matches = sum(
    ref == pred
    for ref, pred in zip(references, hypotheses)
)

# Save evaluation results
evaluation_summary = {
    "model": "Whisper Small LoRA",
    "samples": len(predictions_df),
    "WER": validation_wer,
    "CER": validation_cer,
    "exact_matches": exact_matches,
    "exact_match_percentage": exact_matches / len(predictions_df) * 100
}

print("===== LoRA VALIDATION RESULTS =====")
print(f"Samples: {len(predictions_df)}")
print(f"WER: {validation_wer:.4f} ({validation_wer * 100:.2f}%)")
print(f"CER: {validation_cer:.4f} ({validation_cer * 100:.2f}%)")
print(f"Exact matches: {exact_matches}")
print(
    f"Exact match percentage: "
    f"{evaluation_summary['exact_match_percentage']:.2f}%"
)

# Save normalized predictions and summary
predictions_df.to_csv(PREDICTION_FILE, index=False)

summary_path = os.path.join(
    RESULTS_DIR, "validation_evaluation_summary.csv"
)

pd.DataFrame([evaluation_summary]).to_csv(
    summary_path, index=False
)

print("\nEvaluation summary saved:", summary_path)

===== LoRA VALIDATION RESULTS =====
Samples: 7622
WER: 0.5429 (54.29%)
CER: 0.1571 (15.71%)
Exact matches: 1330
Exact match percentage: 17.45%

Evaluation summary saved: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_lora/results/validation_evaluation_summary.csv


In [ ]:
# Show 20 validation examples
display(
    predictions_df[
        ["reference", "prediction", "ref_clean", "pred_clean"]
    ].sample(20, random_state=42)
)

,reference,prediction,ref_clean,pred_clean
1321,ठाउँ ठाउँमा मन्दिरहरू,ठाउँठाउमा मन्दिरहरू,ठाउँ ठाउँमा मन्दिरहरू,ठाउँठाउमा मन्दिरहरू
3520,औषधि निर्माता कम्पनी,अउचादि निर्माता कम्पनी,औषधि निर्माता कम्पनी,अउचादि निर्माता कम्पनी
3176,गुनासो आदि भएमा,गुनासो आदि भएमा,गुनासो आदि भएमा,गुनासो आदि भएमा
132,तथा अनुसन्धानलाई चार,तथा अनुसन्नालाई चार,तथा अनुसन्धानलाई चार,तथा अनुसन्नालाई चार
1430,मानिने गुरु गोरखनाथको,मानिने गुरु गुरकनातको,मानिने गुरु गोरखनाथको,मानिने गुरु गुरकनातको
457,एपीएफका गोलकिपर समित राजभण्डारीको त्यस बललाई र...,एपियणका गोलकिपर समित राजबन्डारीको त्यसबललाई रो...,एपीएफका गोलकिपर समित राजभण्डारीको त्यस बललाई र...,एपियणका गोलकिपर समित राजबन्डारीको त्यसबललाई रो...
1768,उपराष्ट्रपतिको पारिश्रमिक,उबराष्ट्रपतिको पारिस्रमेक,उपराष्ट्रपतिको पारिश्रमिक,उबराष्ट्रपतिको पारिस्रमेक
4641,उद्धार गर्न खोजे,उद्धार गर्न खोजे,उद्धार गर्न खोजे,उद्धार गर्न खोजे
5405,घमण्डी र सामान्तिको,गवण्डी र सामान्तीको,घमण्डी र सामान्तिको,गवण्डी र सामान्तीको
7407,छोप्नैका लागि उनले,छोप्नैका लागि उनले,छोप्नैका लागि उनले,छोप्नैका लागि उनले


In [ ]:
import pandas as pd
import os

TRAINER_STATE_PATH = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/checkpoints/checkpoint-2000/trainer_state.json"
)

import json

with open(TRAINER_STATE_PATH, "r") as f:
    trainer_state = json.load(f)

loss_df = pd.DataFrame(trainer_state["log_history"])

print("Training log entries:", len(loss_df))

display(
    loss_df[
        [col for col in ["step", "loss", "learning_rate", "epoch"]
         if col in loss_df.columns]
    ].tail(20)
)

Training log entries: 20


,step,loss,learning_rate,epoch
0,100,9.822075,0.000099,0.045790
1,200,4.999469,0.000095,0.091580
2,300,4.166577,0.000090,0.137370
3,400,3.731791,0.000086,0.183161
4,500,3.414131,0.000081,0.228951
5,600,3.184703,0.000076,0.274741
6,700,2.985196,0.000071,0.320531
7,800,2.920942,0.000066,0.366321
8,900,2.875977,0.000062,0.412111
9,1000,2.780210,0.000057,0.457902


## 10. Prepare and resume test inference

The test predictions were generated in more than one runtime session. These cells inspect the existing prediction file, identify which test samples are still missing, reload the LoRA model, and continue inference from the remaining samples.

The earlier full test-inference cell and the repeated `jiwer` installation cell have been removed. The resume workflow below uses the saved predictions rather than starting test inference again from the beginning.

In [ ]:
import os
import pandas as pd

OUTPUT_CSV = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/results/test_predictions.csv"
)

if os.path.exists(OUTPUT_CSV):
    saved_df = pd.read_csv(OUTPUT_CSV)

    print("Saved predictions:", len(saved_df))
    print("Expected test samples:", 7495)
    print("Remaining samples:", 7495 - len(saved_df))
    print("\nLast saved predictions:")
    display(saved_df.tail())
else:
    print("No checkpoint file found at:", OUTPUT_CSV)

Saved predictions: 2400
Expected test samples: 7495
Remaining samples: 5095

Last saved predictions:


,file_id,reference,prediction
2395,51ce8d2192,उहाँको आत्मकथाको पुस्तक,उहाँको आत्मकथाको पुस्तक
2396,518aedc7d8,विवरण गरिएको छ,पिभराण गर्यको छ
2397,51eeba241a,बनाइएको नशा लाग्ने,बन्न येकोर नसालाग्ने
2398,5191e6e959,निर्वाचनको बेला उहाँ,निर्वाचनको बेला उहाँ
2399,51b290102b,पत्रपत्रिकाहरूमा दैनिक रूपमा,पत्रपत्रीकाहरूमा दैनिक रूपमा


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Nepali_Whisper_Project"

TEST_CSV = os.path.join(PROJECT_DIR, "test_manifest.csv")
TEST_AUDIO_DIR = os.path.join(PROJECT_DIR, "audio", "test")

OUTPUT_CSV = os.path.join(
    PROJECT_DIR,
    "whisper_small_lora",
    "results",
    "test_predictions.csv"
)

test_df = pd.read_csv(TEST_CSV)

test_df["audio_path"] = test_df["file_id"].apply(
    lambda file_id: os.path.join(
        TEST_AUDIO_DIR, f"{file_id}.flac"
    )
)

saved_df = pd.read_csv(OUTPUT_CSV)

completed_ids = set(saved_df["file_id"].astype(str))

remaining_df = test_df[
    ~test_df["file_id"].astype(str).isin(completed_ids)
].copy()

print("Total test samples:", len(test_df))
print("Previously completed:", len(saved_df))
print("Remaining samples:", len(remaining_df))
print("Missing audio files:",
      (~remaining_df["audio_path"].apply(os.path.exists)).sum())

assert len(saved_df) == len(completed_ids), "Duplicate saved file IDs!"
assert len(test_df) == 7495
assert len(remaining_df) == 5095
assert remaining_df["audio_path"].apply(os.path.exists).all()

print("\nReady to resume inference.")

Total test samples: 7495
Previously completed: 2400
Remaining samples: 5095
Missing audio files: 0

Ready to resume inference.


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

BASE_MODEL = "openai/whisper-small"

ADAPTER_DIR = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/final_model"
)

# Load processor
processor = WhisperProcessor.from_pretrained(BASE_MODEL)

# Load original pretrained Whisper Small
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)

# Attach trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR
)

model = model.to("cuda")
model.eval()

print("LoRA model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

LoRA model loaded successfully!
GPU: Tesla T4


In [ ]:
import time
import librosa
import torch
import pandas as pd
from tqdm.auto import tqdm

BATCH_SIZE = 4
SAVE_EVERY = 50

# Start with existing predictions
test_predictions = saved_df.to_dict("records")

completed_ids = set(saved_df["file_id"].astype(str))

remaining_df = test_df[
    ~test_df["file_id"].astype(str).isin(completed_ids)
].copy()

model.eval()

start_time = time.time()
new_count = 0

for start_idx in tqdm(
    range(0, len(remaining_df), BATCH_SIZE),
    desc="Resuming LoRA Test Inference"
):
    batch_df = remaining_df.iloc[
        start_idx:start_idx + BATCH_SIZE
    ]

    audio_batch = []

    for audio_path in batch_df["audio_path"]:
        audio, sr = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )
        audio_batch.append(audio)

    inputs = processor(
        audio_batch,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device="cuda",
        dtype=torch.float16
    )

    with torch.inference_mode():
        predicted_ids = model.generate(
            input_features,
            language="ne",
            task="transcribe",
            use_cache=True
        )

    predictions = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )

    for file_id, reference, prediction in zip(
        batch_df["file_id"],
        batch_df["transcription"],
        predictions
    ):
        test_predictions.append({
            "file_id": file_id,
            "reference": reference,
            "prediction": prediction
        })

    new_count += len(batch_df)

    if new_count >= SAVE_EVERY and (
        new_count % SAVE_EVERY < BATCH_SIZE
        or new_count == len(remaining_df)
    ):
        pd.DataFrame(test_predictions).to_csv(
            OUTPUT_CSV,
            index=False
        )

# Final save
test_predictions_df = pd.DataFrame(test_predictions)
test_predictions_df.to_csv(OUTPUT_CSV, index=False)

elapsed = time.time() - start_time

print("\n===== RESUMED LoRA INFERENCE COMPLETED =====")
print("New predictions:", new_count)
print("Total predictions:", len(test_predictions_df))
print("Inference time:", round(elapsed / 60, 2), "minutes")
print("Average time/sample:", round(elapsed / new_count, 3), "seconds")
print("Throughput:", round(new_count / elapsed, 2), "samples/second")
print("Saved:", OUTPUT_CSV)

Resuming LoRA Test Inference:   0%|          | 0/1274 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



===== RESUMED LoRA INFERENCE COMPLETED =====
New predictions: 5095
Total predictions: 7495
Inference time: 53.08 minutes
Average time/sample: 0.625 seconds
Throughput: 1.6 samples/second
Saved: /content/drive/MyDrive/Nepali_Whisper_Project/whisper_small_lora/results/test_predictions.csv


## 11. Evaluate the test set

The saved test predictions are loaded and normalized using the same function as the validation evaluation. WER, CER, and exact-match results are then calculated for the test set.

In [ ]:
import pandas as pd
import re
import unicodedata
from jiwer import wer, cer

OUTPUT_CSV = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/results/test_predictions.csv"
)

df = pd.read_csv(OUTPUT_CSV)

def normalize_text(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["ref_clean"] = df["reference"].apply(normalize_text)
df["pred_clean"] = df["prediction"].apply(normalize_text)

test_wer = wer(
    df["ref_clean"].tolist(),
    df["pred_clean"].tolist()
)

test_cer = cer(
    df["ref_clean"].tolist(),
    df["pred_clean"].tolist()
)

exact_matches = (
    df["ref_clean"] == df["pred_clean"]
).sum()

print("===== LoRA TEST EVALUATION =====")
print("Test samples:", len(df))
print(f"WER: {test_wer * 100:.2f}%")
print(f"CER: {test_cer * 100:.2f}%")
print(f"Exact matches: {exact_matches}")
print(f"Exact match rate: {exact_matches / len(df) * 100:.2f}%")

===== LoRA TEST EVALUATION =====
Test samples: 7495
WER: 52.74%
CER: 14.96%
Exact matches: 1483
Exact match rate: 19.79%


In [ ]:
from jiwer import process_words
import numpy as np

sample_wers = []

for _, row in df.iterrows():
    ref = row["ref_clean"]
    pred = row["pred_clean"]

    if not ref.strip():
        sample_wer = 0 if not pred.strip() else 1
    else:
        sample_wer = process_words(ref, pred).wer

    sample_wers.append(sample_wer)

df["sample_wer"] = sample_wers

print("===== WORD-LEVEL ERROR DISTRIBUTION =====")
print("Total samples:", len(df))
print("Exact matches:", (df["sample_wer"] == 0).sum())

for threshold in [1, 2, 3, 5, 10]:
    count = (df["sample_wer"] >= threshold).sum()
    print(
        f"WER >= {threshold}: {count} samples "
        f"({count / len(df) * 100:.2f}%)"
    )

print("\n===== SAMPLE WER STATISTICS =====")
print(
    df["sample_wer"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

===== WORD-LEVEL ERROR DISTRIBUTION =====
Total samples: 7495
Exact matches: 1483
WER >= 1: 1660 samples (22.15%)
WER >= 2: 19 samples (0.25%)
WER >= 3: 1 samples (0.01%)
WER >= 5: 0 samples (0.00%)
WER >= 10: 0 samples (0.00%)

===== SAMPLE WER STATISTICS =====
count    7495.000000
mean        0.529159
std         0.378295
min         0.000000
25%         0.333333
50%         0.500000
75%         0.666667
90%         1.000000
95%         1.000000
99%         1.500000
max         3.000000
Name: sample_wer, dtype: float64


In [ ]:
from jiwer import process_characters

sample_cers = []

for _, row in df.iterrows():
    ref = row["ref_clean"]
    pred = row["pred_clean"]

    if not ref.strip():
        sample_cer = 0 if not pred.strip() else 1
    else:
        sample_cer = process_characters(
            ref, pred
        ).cer

    sample_cers.append(sample_cer)

df["sample_cer"] = sample_cers

print("===== CHARACTER-LEVEL ERROR DISTRIBUTION =====")
print("Total samples:", len(df))
print("Exact matches:", (df["sample_cer"] == 0).sum())

for threshold in [0.25, 0.50, 0.75, 1, 2]:
    count = (df["sample_cer"] >= threshold).sum()
    print(
        f"CER >= {threshold}: {count} samples "
        f"({count / len(df) * 100:.2f}%)"
    )

print("\n===== SAMPLE CER STATISTICS =====")
print(
    df["sample_cer"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

===== CHARACTER-LEVEL ERROR DISTRIBUTION =====
Total samples: 7495
Exact matches: 1483
CER >= 0.25: 1359 samples (18.13%)
CER >= 0.5: 167 samples (2.23%)
CER >= 0.75: 43 samples (0.57%)
CER >= 1: 18 samples (0.24%)
CER >= 2: 6 samples (0.08%)

===== SAMPLE CER STATISTICS =====
count    7495.000000
mean        0.146203
std         0.221420
min         0.000000
25%         0.050000
50%         0.111111
75%         0.200000
90%         0.307692
95%         0.384615
99%         0.625395
max        12.941176
Name: sample_cer, dtype: float64


In [ ]:
# Sort by character-level error
worst_samples = df.sort_values(
    by="sample_cer",
    ascending=False
).head(20)

print("===== 20 WORST LoRA PREDICTIONS =====")

display(
    worst_samples[
        [
            "file_id",
            "reference",
            "prediction",
            "sample_wer",
            "sample_cer"
        ]
    ].reset_index(drop=True)
)

===== 20 WORST LoRA PREDICTIONS =====


,file_id,reference,prediction,sample_wer,sample_cer
0,3ce03cadc9,९ सेप्टेम्बर १९८७,१९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९...,1.000000,12.941176
1,baad9a596c,ग्रीष्‍म कालमा नै यहाँ धेरै पर्यटक आउँदछन्।,गृष्मा कालमा नै यहाँ धेरे परेटक आउँँँँँँँँँँँँ...,0.571429,5.000000
2,e2717db05e,यस पुरस्कारको राशि मदन पुरस्कार सरह अर्थात् रु...,यस पृष्कारको राशीमधन पृषकार सरह अर्थात रू २०२०...,0.777778,3.849057
3,d53f8dcb3e,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,नेपालको सन् २०००१०१०००००००००००००००००००००००००००...,0.800000,3.459016
4,74081b2c36,नेपालको सन् २०११ को जनगणना अनुसार च्यामाको जनस...,नेपालको सन् २०००१०० गर्देना अनुसर च्यामाको जनस...,0.636364,3.078125
5,3d1ab2b4bb,र,र हो,1.000000,3.000000
6,43f3281d7a,सन् १९५८ १९६२,सनुन्नैसे अन्ठाउन्न १९८,1.000000,1.307692
7,c07c264601,१९९३ र फरेस्ट,उन्यसेतिरे नभि र फरेश,1.000000,1.307692
8,557499d1f5,कोड ०४७ हो,कोड जेरो चार साथ हो,1.000000,1.200000
9,6ca0956b3c,ईपू ८८२ ईसा,इपू आर्शै बयासी इसा,1.333333,1.181818


## 12. Compare LoRA with full fine-tuning

The full fine-tuning (FFT) predictions are loaded and matched with the LoRA predictions using `file_id`. The reference transcripts are checked for consistency before comparing sample-level WER.

The analysis also looks at performance by reference length and checks whether samples containing Devanagari numbers are more difficult for either model.

In [ ]:
from jiwer import process_words, process_characters

FFT_CSV = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_fft/full_training/results/test_predictions.csv"
)

fft_df = pd.read_csv(FFT_CSV)

# Normalize FFT predictions using the same function
fft_df["ref_clean"] = fft_df["reference"].apply(normalize_text)
fft_df["fft_clean"] = fft_df["prediction"].apply(normalize_text)

# Select the same 20 worst LoRA samples
comparison = worst_samples[
    ["file_id", "reference", "prediction",
     "sample_wer", "sample_cer"]
].merge(
    fft_df[["file_id", "prediction", "ref_clean", "fft_clean"]],
    on="file_id",
    how="left",
    suffixes=("_lora", "_fft")
)

# Check reference consistency
comparison["reference_matches"] = (
    comparison["ref_clean"] == comparison["reference"].apply(normalize_text)
)

# Calculate FFT sample-level WER
comparison["fft_wer"] = comparison.apply(
    lambda row: process_words(
        row["ref_clean"], row["fft_clean"]
    ).wer if row["ref_clean"].strip()
    else (0 if not row["fft_clean"].strip() else 1),
    axis=1
)

print("FFT predictions missing:", comparison["fft_clean"].isna().sum())
print("Reference mismatches:", (~comparison["reference_matches"]).sum())

display(
    comparison[
        [
            "file_id",
            "reference",
            "prediction_lora",
            "fft_clean",
            "sample_wer",
            "fft_wer"
        ]
    ].reset_index(drop=True)
)

FFT predictions missing: 0
Reference mismatches: 0


,file_id,reference,prediction_lora,fft_clean,sample_wer,fft_wer
0,3ce03cadc9,९ सेप्टेम्बर १९८७,१९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९...,१९७७७,1.000000,1.000000
1,baad9a596c,ग्रीष्‍म कालमा नै यहाँ धेरै पर्यटक आउँदछन्।,गृष्मा कालमा नै यहाँ धेरे परेटक आउँँँँँँँँँँँँ...,गृष्मकालमा नै यहाँ धेरै पर्यटक आउँचछन्,0.571429,0.428571
2,e2717db05e,यस पुरस्कारको राशि मदन पुरस्कार सरह अर्थात् रु...,यस पृष्कारको राशीमधन पृषकार सरह अर्थात रू २०२०...,यस पुरुष्कारको राशिमदन पुरुष्कार सरह अर्थात रू...,0.777778,0.888889
3,d53f8dcb3e,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,नेपालको सन् २०००१०१०००००००००००००००००००००००००००...,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,0.800000,0.000000
4,74081b2c36,नेपालको सन् २०११ को जनगणना अनुसार च्यामाको जनस...,नेपालको सन् २०००१०० गर्देना अनुसर च्यामाको जनस...,नेपालको सन् २०११ को जनगणेना अनुसार च्यामाको जन...,0.636364,0.181818
5,3d1ab2b4bb,र,र हो,र हो,1.000000,1.000000
6,43f3281d7a,सन् १९५८ १९६२,सनुन्नैसे अन्ठाउन्न १९८,सन् १९८८ १९६२,1.000000,0.333333
7,c07c264601,१९९३ र फरेस्ट,उन्यसेतिरे नभि र फरेश,१९३ र फरेष्ट,1.000000,0.666667
8,557499d1f5,कोड ०४७ हो,कोड जेरो चार साथ हो,कोड ००४ साथ हो,1.000000,0.666667
9,6ca0956b3c,ईपू ८८२ ईसा,इपू आर्शै बयासी इसा,ईपू ८८२ ईसा,1.333333,0.000000


In [ ]:
# Merge all LoRA and FFT predictions by file_id
full_comparison = df[
    ["file_id", "ref_clean", "pred_clean", "sample_wer"]
].merge(
    fft_df[["file_id", "ref_clean", "fft_clean"]],
    on="file_id",
    how="inner",
    suffixes=("_lora", "_fft")
)

# Confirm references match
assert (
    full_comparison["ref_clean_lora"]
    == full_comparison["ref_clean_fft"]
).all(), "Reference mismatch detected!"

# Calculate FFT sample-level WER
full_comparison["fft_wer"] = full_comparison.apply(
    lambda row: process_words(
        row["ref_clean_fft"], row["fft_clean"]
    ).wer if row["ref_clean_fft"].strip()
    else (0 if not row["fft_clean"].strip() else 1),
    axis=1
)

# Compare sample-level performance
lora_wer = full_comparison["sample_wer"]
fft_wer = full_comparison["fft_wer"]

lora_better = (lora_wer < fft_wer).sum()
fft_better = (fft_wer < lora_wer).sum()
equal = (lora_wer == fft_wer).sum()

print("===== PAIRED SAMPLE-LEVEL COMPARISON =====")
print("Matched samples:", len(full_comparison))
print(f"LoRA better: {lora_better} ({lora_better/len(df)*100:.2f}%)")
print(f"FFT better: {fft_better} ({fft_better/len(df)*100:.2f}%)")
print(f"Equal: {equal} ({equal/len(df)*100:.2f}%)")

print("\n===== EXACT MATCH COMPARISON =====")
print("LoRA exact:", (lora_wer == 0).sum())
print("FFT exact:", (fft_wer == 0).sum())

print("\n===== MEAN SAMPLE WER =====")
print("LoRA:", round(lora_wer.mean(), 4))
print("FFT:", round(fft_wer.mean(), 4))

===== PAIRED SAMPLE-LEVEL COMPARISON =====
Matched samples: 7495
LoRA better: 378 (5.04%)
FFT better: 3967 (52.93%)
Equal: 3150 (42.03%)

===== EXACT MATCH COMPARISON =====
LoRA exact: 1483
FFT exact: 3619

===== MEAN SAMPLE WER =====
LoRA: 0.5292
FFT: 0.2802


In [ ]:
# Calculate reference word count
full_comparison["reference_word_count"] = (
    full_comparison["ref_clean_lora"].str.split().str.len()
)

# Group recordings by reference length
full_comparison["length_group"] = pd.cut(
    full_comparison["reference_word_count"],
    bins=[-1, 5, 10, 20, 50, float("inf")],
    labels=["1-5", "6-10", "11-20", "21-50", "50+"]
)

length_analysis = full_comparison.groupby(
    "length_group", observed=False
).agg(
    samples=("file_id", "count"),
    lora_mean_wer=("sample_wer", "mean"),
    fft_mean_wer=("fft_wer", "mean"),
    lora_exact=("sample_wer", lambda x: (x == 0).sum()),
    fft_exact=("fft_wer", lambda x: (x == 0).sum())
).reset_index()

length_analysis["lora_exact_rate"] = (
    length_analysis["lora_exact"] /
    length_analysis["samples"] * 100
)

length_analysis["fft_exact_rate"] = (
    length_analysis["fft_exact"] /
    length_analysis["samples"] * 100
)

display(length_analysis.round(4))

,length_group,samples,lora_mean_wer,fft_mean_wer,lora_exact,fft_exact,lora_exact_rate,fft_exact_rate
0,1-5,7376,0.5282,0.2791,1482,3609,20.0922,48.9290
1,6-10,79,0.5668,0.3357,1,10,1.2658,12.6582
2,11-20,40,0.6312,0.3758,0,0,0.0000,0.0000
3,21-50,0,NaN,NaN,0,0,NaN,NaN
4,50+,0,NaN,NaN,0,0,NaN,NaN


In [ ]:
import re

# Detect Devanagari digits in reference text
digit_pattern = re.compile(r"[०-९]")

full_comparison["contains_numbers"] = (
    full_comparison["ref_clean_lora"].apply(
        lambda x: bool(digit_pattern.search(x))
    )
)

numeric_df = full_comparison[
    full_comparison["contains_numbers"]
]

non_numeric_df = full_comparison[
    ~full_comparison["contains_numbers"]
]

print("===== NUMERICAL CONTENT ANALYSIS =====")
print("Samples containing numbers:", len(numeric_df))
print("Samples without numbers:", len(non_numeric_df))

print("\n===== MEAN SAMPLE WER =====")
print("Numeric - LoRA:", numeric_df["sample_wer"].mean())
print("Numeric - FFT:", numeric_df["fft_wer"].mean())

print("\nNon-numeric - LoRA:", non_numeric_df["sample_wer"].mean())
print("Non-numeric - FFT:", non_numeric_df["fft_wer"].mean())

print("\n===== EXACT MATCH RATES =====")
print(
    "Numeric - LoRA:",
    (numeric_df["sample_wer"] == 0).mean() * 100
)
print(
    "Numeric - FFT:",
    (numeric_df["fft_wer"] == 0).mean() * 100
)

print(
    "Non-numeric - LoRA:",
    (non_numeric_df["sample_wer"] == 0).mean() * 100
)
print(
    "Non-numeric - FFT:",
    (non_numeric_df["fft_wer"] == 0).mean() * 100
)

===== NUMERICAL CONTENT ANALYSIS =====
Samples containing numbers: 481
Samples without numbers: 7014

===== MEAN SAMPLE WER =====
Numeric - LoRA: 0.7028358451435374
Numeric - FFT: 0.357793430870354

Non-numeric - LoRA: 0.5172484119340407
Non-numeric - FFT: 0.27485531529586277

===== EXACT MATCH RATES =====
Numeric - LoRA: 1.2474012474012475
Numeric - FFT: 35.550935550935556
Non-numeric - LoRA: 21.057884231536928
Non-numeric - FFT: 49.158825206729404


In [ ]:
import re

numeric_examples = numeric_df.copy()

numeric_examples["number_tokens"] = (
    numeric_examples["ref_clean_lora"].apply(
        lambda text: re.findall(r"[०-९]+", text)
    )
)

# Show samples where either model made an error
numeric_examples["both_wrong"] = (
    (numeric_examples["sample_wer"] > 0) &
    (numeric_examples["fft_wer"] > 0)
)

numeric_examples["lora_only_wrong"] = (
    (numeric_examples["sample_wer"] > 0) &
    (numeric_examples["fft_wer"] == 0)
)

print("Total numerical samples:", len(numeric_examples))
print("Both models incorrect:", numeric_examples["both_wrong"].sum())
print("Only LoRA incorrect:", numeric_examples["lora_only_wrong"].sum())

print("\n===== WORST NUMERICAL LoRA EXAMPLES =====")

display(
    numeric_examples.sort_values(
        "sample_wer", ascending=False
    )[
        [
            "file_id",
            "ref_clean_lora",
            "pred_clean",
            "fft_clean",
            "number_tokens",
            "sample_wer",
            "fft_wer"
        ]
    ].head(25).reset_index(drop=True)
)

Total numerical samples: 481
Both models incorrect: 309
Only LoRA incorrect: 166

===== WORST NUMERICAL LoRA EXAMPLES =====


,file_id,ref_clean_lora,pred_clean,fft_clean,number_tokens,sample_wer,fft_wer
0,68bf24e890,१५० किलोमिटर,१०५ किलो मिटर,१५० किलोमिटर,[१५०],1.500000,0.000000
1,10ed23ec08,३८७ त्यसलाई बेचे,१२०० मलव २१० वेचे,३८ दशमलव ७ तेसलाई बेचे,[३८७],1.333333,1.333333
2,21f1912efe,स्टेङ्गोद्वारा डिसेम्बर २००९,स्टेङ्गो द्वारा दिसेम्बा २०००,स्टेङ्गोद्वारा डिसेम्बर २००९,[२००९],1.333333,0.000000
3,aaef74abde,जीवनप्रेम श्रेष्ठ २०२६,जीवन प्रेम स्रेष्ट २०२,जीवन प्रेमश्रेष्ट २०२६,[२०२६],1.333333,0.666667
4,215c83d1ba,२०६७ नेपाली साहित्यकार,२०० सर्षति निपाली साहितेका,२००६ नेपाली साहित्यका,[२०६७],1.333333,0.666667
5,f6f22f35aa,गराउने उद्देश्यअनुरूप ०२७,घराउने उदेश्य अन्रूप २०००,गोराउने उद्देश्यअन्द्रूप १६७,[०२७],1.333333,1.000000
6,7f746e3e15,अस्ट्रेलियादेखि लगभग २,अस्टेल्या देखि लगबग दुई,अस्ट्रेलियादेखि लगभग २,[२],1.333333,0.000000
7,6ca0956b3c,ईपू ८८२ ईसा,इपू आर्शै बयासी इसा,ईपू ८८२ ईसा,[८८२],1.333333,0.000000
8,6361aff6f0,यहाँ ३५० मानिसहरूलाई,याहा २००५ मानी सहरूलाई,यहाँ ३५० मानिसहरूलाई,[३५०],1.333333,0.000000
9,a0cf67af10,घिउमा ८ मिनेटसम्म,ज्युमा आठ मिनेट सम्म,घ्यूमा आठ मिनेट सम्म,[८],1.333333,1.333333


In [ ]:
ANALYSIS_DIR = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "whisper_small_lora/results"
)

# Save full paired comparison
full_comparison.to_csv(
    f"{ANALYSIS_DIR}/fft_vs_lora_sample_comparison.csv",
    index=False
)

# Save numerical subset
numeric_examples.to_csv(
    f"{ANALYSIS_DIR}/lora_numerical_error_analysis.csv",
    index=False
)

# Save length analysis
length_analysis.to_csv(
    f"{ANALYSIS_DIR}/lora_length_error_analysis.csv",
    index=False
)

print("Analysis files saved successfully.")

Analysis files saved successfully.


## 13. Repetition and speaker-level error analysis

These checks look for repeated characters and repeated words in the reference transcripts and model predictions. They help identify cases where a model produces repeated text.

The final analysis groups the results by speaker to see whether recognition errors differ across speakers. The resulting comparison files are saved in the experiment's results folder.

In [ ]:
import re
from itertools import groupby

def max_character_run(text):
    """Find the longest consecutive identical-character sequence."""
    if not isinstance(text, str) or not text:
        return 0, ""

    best_length = 0
    best_char = ""

    for char, group in groupby(text):
        length = sum(1 for _ in group)

        if not char.isspace() and length > best_length:
            best_length = length
            best_char = char

    return best_length, best_char


def max_word_run(text):
    """Find the longest consecutive repetition of an identical word."""
    if not isinstance(text, str) or not text.strip():
        return 0, ""

    words = text.split()
    best_length = 0
    best_word = ""

    for word, group in groupby(words):
        length = sum(1 for _ in group)

        if length > best_length:
            best_length = length
            best_word = word

    return best_length, best_word


# Analyze reference, LoRA and FFT outputs
for name, column in [
    ("reference", "ref_clean_lora"),
    ("lora", "pred_clean"),
    ("fft", "fft_clean")
]:
    full_comparison[f"{name}_char_run"] = (
        full_comparison[column].apply(
            lambda x: max_character_run(x)[0]
        )
    )

    full_comparison[f"{name}_char_pattern"] = (
        full_comparison[column].apply(
            lambda x: max_character_run(x)[1]
        )
    )

    full_comparison[f"{name}_word_run"] = (
        full_comparison[column].apply(
            lambda x: max_word_run(x)[0]
        )
    )

    full_comparison[f"{name}_word_pattern"] = (
        full_comparison[column].apply(
            lambda x: max_word_run(x)[1]
        )
    )


print("===== REPETITION ANALYSIS =====")

for name in ["reference", "lora", "fft"]:
    char_count = (
        full_comparison[f"{name}_char_run"] >= 3
    ).sum()

    word_count = (
        full_comparison[f"{name}_word_run"] >= 2
    ).sum()

    print(f"\n{name.upper()}")
    print(f"Character repetition (3+): {char_count}")
    print(f"Word repetition (2+): {word_count}")

===== REPETITION ANALYSIS =====

REFERENCE
Character repetition (3+): 8
Word repetition (2+): 16

LORA
Character repetition (3+): 92
Word repetition (2+): 18

FFT
Character repetition (3+): 16
Word repetition (2+): 18


In [ ]:
repetition_examples = full_comparison[
    (full_comparison["lora_char_run"] >= 3) |
    (full_comparison["lora_word_run"] >= 2)
].copy()

repetition_examples = repetition_examples.sort_values(
    by=["lora_char_run", "lora_word_run"],
    ascending=False
)

display(
    repetition_examples[
        [
            "file_id",
            "ref_clean_lora",
            "pred_clean",
            "fft_clean",
            "lora_char_run",
            "lora_char_pattern",
            "lora_word_run",
            "lora_word_pattern"
        ]
    ].head(25).reset_index(drop=True)
)

,file_id,ref_clean_lora,pred_clean,fft_clean,lora_char_run,lora_char_pattern,lora_word_run,lora_word_pattern
0,3ce03cadc9,९ सेप्टेम्बर १९८७,१९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९...,१९७७७,221,९,1,१९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९...
1,d53f8dcb3e,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,नेपालको सन् २०००१०१०००००००००००००००००००००००००००...,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,209,०,1,नेपालको
2,baad9a596c,ग्रीष्‍म कालमा नै यहाँ धेरै पर्यटक आउँदछन्,गृष्मा कालमा नै यहाँ धेरे परेटक आउँँँँँँँँँँँँ...,गृष्मकालमा नै यहाँ धेरै पर्यटक आउँचछन्,202,ँ,1,गृष्मा
3,74081b2c36,नेपालको सन् २०११ को जनगणना अनुसार च्यामाको जनस...,नेपालको सन् २०००१०० गर्देना अनुसर च्यामाको जनस...,नेपालको सन् २०११ को जनगणेना अनुसार च्यामाको जन...,187,०,1,नेपालको
4,e2717db05e,यस पुरस्कारको राशि मदन पुरस्कार सरह अर्थात् रु...,यस पृष्कारको राशीमधन पृषकार सरह अर्थात रू २०२०...,यस पुरुष्कारको राशिमदन पुरुष्कार सरह अर्थात रू...,153,०,1,यस
5,0496cd16ce,१७०००० सेना रहेको देखियो,१००००००००००० सिनारहेको दिख्यो,एक साथ सुन्य सुन्य सुन्य सेनारहेको देख्यो,11,०,1,१०००००००००००
6,b34fbf1880,पर्यटन वर्ष २०११,परेटन वर्षु २०००००००००००,पर्यटन वर्ष २०११,11,०,1,परेटन
7,8528be4dcd,यसमध्ये पुरुष ५१ प्रतिशत र महिला ४९ प्रतिशत छन्,यसमत्य पुरुस १००००००००००,यसमत्य पुरुष ५१ प्रतिशत्र महिला ५९ प्रतिशत्छन्,10,०,1,यसमत्य
8,af5c926715,१९९१ को डिसेम्बरमा,१९९९९९९९९९९,१९९० को डिसेम्बरमा,10,९,1,१९९९९९९९९९९
9,c7f17b8829,२०५८ को राष्ट्रिय,२००००००००००,२०५८ को राष्ट्रिय,10,०,1,२००००००००००


In [ ]:
# Select all LoRA samples with repeated characters
lora_repetition = full_comparison[
    full_comparison["lora_char_run"] >= 3
].copy()

# Add FFT repetition information
lora_repetition["fft_has_repetition"] = (
    lora_repetition["fft_char_run"] >= 3
)

lora_repetition["reference_has_repetition"] = (
    lora_repetition["reference_char_run"] >= 3
)

print("===== LoRA REPETITION BREAKDOWN =====")
print("Total LoRA repetition samples:", len(lora_repetition))

print(
    "Also repeated in FFT:",
    lora_repetition["fft_has_repetition"].sum()
)

print(
    "Also repeated in reference:",
    lora_repetition["reference_has_repetition"].sum()
)

print(
    "LoRA-only repetition:",
    (
        ~lora_repetition["fft_has_repetition"] &
        ~lora_repetition["reference_has_repetition"]
    ).sum()
)

print("\n===== REPETITION EXAMPLES =====")

display(
    lora_repetition[
        [
            "file_id",
            "ref_clean_lora",
            "pred_clean",
            "fft_clean",
            "lora_char_run",
            "lora_char_pattern",
            "fft_char_run"
        ]
    ].sort_values(
        "lora_char_run",
        ascending=False
    ).head(30).reset_index(drop=True)
)

===== LoRA REPETITION BREAKDOWN =====
Total LoRA repetition samples: 92
Also repeated in FFT: 9
Also repeated in reference: 6
LoRA-only repetition: 81

===== REPETITION EXAMPLES =====


,file_id,ref_clean_lora,pred_clean,fft_clean,lora_char_run,lora_char_pattern,fft_char_run
0,3ce03cadc9,९ सेप्टेम्बर १९८७,१९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९९...,१९७७७,221,९,3
1,d53f8dcb3e,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,नेपालको सन् २०००१०१०००००००००००००००००००००००००००...,नेपालको सन् २००१ को जनगणना अनुसार चौलाखर्कको ज...,209,०,2
2,baad9a596c,ग्रीष्‍म कालमा नै यहाँ धेरै पर्यटक आउँदछन्,गृष्मा कालमा नै यहाँ धेरे परेटक आउँँँँँँँँँँँँ...,गृष्मकालमा नै यहाँ धेरै पर्यटक आउँचछन्,202,ँ,1
3,74081b2c36,नेपालको सन् २०११ को जनगणना अनुसार च्यामाको जनस...,नेपालको सन् २०००१०० गर्देना अनुसर च्यामाको जनस...,नेपालको सन् २०११ को जनगणेना अनुसार च्यामाको जन...,187,०,2
4,e2717db05e,यस पुरस्कारको राशि मदन पुरस्कार सरह अर्थात् रु...,यस पृष्कारको राशीमधन पृषकार सरह अर्थात रू २०२०...,यस पुरुष्कारको राशिमदन पुरुष्कार सरह अर्थात रू...,153,०,3
5,0496cd16ce,१७०००० सेना रहेको देखियो,१००००००००००० सिनारहेको दिख्यो,एक साथ सुन्य सुन्य सुन्य सेनारहेको देख्यो,11,०,1
6,b34fbf1880,पर्यटन वर्ष २०११,परेटन वर्षु २०००००००००००,पर्यटन वर्ष २०११,11,०,2
7,c7f17b8829,२०५८ को राष्ट्रिय,२००००००००००,२०५८ को राष्ट्रिय,10,०,1
8,af5c926715,१९९१ को डिसेम्बरमा,१९९९९९९९९९९,१९९० को डिसेम्बरमा,10,९,2
9,8528be4dcd,यसमध्ये पुरुष ५१ प्रतिशत र महिला ४९ प्रतिशत छन्,यसमत्य पुरुस १००००००००००,यसमत्य पुरुष ५१ प्रतिशत्र महिला ५९ प्रतिशत्छन्,10,०,1


In [ ]:
TEST_CSV = (
    "/content/drive/MyDrive/Nepali_Whisper_Project/"
    "test_manifest.csv"
)

manifest = pd.read_csv(TEST_CSV)

speaker_comparison = full_comparison.merge(
    manifest[["file_id", "speaker_id"]],
    on="file_id",
    how="left"
)

speaker_analysis = speaker_comparison.groupby(
    "speaker_id"
).agg(
    samples=("file_id", "count"),
    lora_mean_wer=("sample_wer", "mean"),
    fft_mean_wer=("fft_wer", "mean"),
    lora_exact=("sample_wer", lambda x: (x == 0).sum()),
    fft_exact=("fft_wer", lambda x: (x == 0).sum())
).reset_index()

speaker_analysis["lora_exact_rate"] = (
    speaker_analysis["lora_exact"] /
    speaker_analysis["samples"] * 100
)

speaker_analysis["fft_exact_rate"] = (
    speaker_analysis["fft_exact"] /
    speaker_analysis["samples"] * 100
)

print("Number of speakers:", len(speaker_analysis))

display(
    speaker_analysis.sort_values(
        "lora_mean_wer",
        ascending=False
    ).round(4)
)

Number of speakers: 20


,speaker_id,samples,lora_mean_wer,fft_mean_wer,lora_exact,fft_exact,lora_exact_rate,fft_exact_rate
16,c09aa,364,0.6772,0.3992,23,108,6.3187,29.6703
5,5f3e5,325,0.6498,0.3583,27,117,8.3077,36.0000
10,8aea1,333,0.6124,0.3685,44,123,13.2132,36.9369
14,9ef53,376,0.6013,0.3130,49,155,13.0319,41.2234
17,d0cd7,403,0.5647,0.3089,73,169,18.1141,41.9355
9,8389b,284,0.5603,0.3304,46,120,16.1972,42.2535
0,07179,354,0.5394,0.2872,66,158,18.6441,44.6328
3,29944,377,0.5391,0.2758,58,179,15.3846,47.4801
2,21ba4,387,0.5237,0.2719,72,198,18.6047,51.1628
13,99866,370,0.5178,0.2895,74,179,20.0000,48.3784


In [ ]:
speaker_analysis.to_csv(
    f"{ANALYSIS_DIR}/lora_speaker_error_analysis.csv",
    index=False
)

print("Speaker analysis saved successfully.")

Speaker analysis saved successfully.
